In [70]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# building the user taste profile, and candidates matrix
dim_tracks = pd.read_csv('../../data/processed/to_db/dim_tracks.csv')
dim_artists = pd.read_csv('../../data/processed/to_db/dim_artists.csv')
fact_interactions = pd.read_csv('../../data/processed/to_db/fact_interactions.csv')
all_artists = pd.read_csv('all_artists.csv')
combined_feats = load_npz('feature_matrix.npz')

# init vectorizers from the previous modeling
tokenizer = lambda s: [t.strip() for t in s.split(",") if t.strip()]

tag_vectorizer = TfidfVectorizer(
    tokenizer=tokenizer, token_pattern=None,
    lowercase=False, sublinear_tf=True, min_df=2
)

similar_artists_vec = TfidfVectorizer(
    tokenizer=tokenizer, token_pattern=None, lowercase=False,
    sublinear_tf=True, min_df=2
)

## Data Inspection

In [71]:
# inspect data
dfs = [dim_tracks, dim_artists, fact_interactions]
str_dfs = ['dim_tracks', 'dim_artists', 'fact_interactions']

print("Number of Duplicates")
for df, name in zip(dfs, str_dfs):
    print(f"Num Dupes for {name}: {df.duplicated().sum()}")

Number of Duplicates
Num Dupes for dim_tracks: 0
Num Dupes for dim_artists: 0
Num Dupes for fact_interactions: 0


In [72]:
print(dim_tracks.isna().sum())

track_id        0
release_date    0
song_name       0
album_name      0
duration_min    0
artists_id      0
dtype: int64


In [73]:
print(dim_artists.isna().sum())

artists_id              0
primary_artist_name     0
artist_listeners        0
artist_playcount        0
tags                   17
similar_artists         0
num_tags                0
num_similar_artists     0
dtype: int64


In [74]:
print(fact_interactions.isna().sum())

interaction_id      0
track_id            0
interaction_type    0
dtype: int64


## Artist Deduplication and Base Signal Mapping

In [75]:
artists_deduped = (
    fact_interactions
    .merge(dim_tracks, how='left', on='track_id')
    .merge(dim_artists, how='left', on='artists_id'))

artists_deduped.info()

<class 'pandas.DataFrame'>
RangeIndex: 866 entries, 0 to 865
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   interaction_id       866 non-null    int64  
 1   track_id             866 non-null    str    
 2   interaction_type     866 non-null    str    
 3   release_date         866 non-null    str    
 4   song_name            866 non-null    str    
 5   album_name           866 non-null    str    
 6   duration_min         866 non-null    float64
 7   artists_id           866 non-null    int64  
 8   primary_artist_name  866 non-null    str    
 9   artist_listeners     866 non-null    int64  
 10  artist_playcount     866 non-null    int64  
 11  tags                 857 non-null    str    
 12  similar_artists      866 non-null    str    
 13  num_tags             866 non-null    int64  
 14  num_similar_artists  866 non-null    int64  
dtypes: float64(1), int64(6), str(8)
memory usage: 467.7

In [76]:
artists_deduped['interaction_type'].value_counts()

interaction_type
Saved              472
Long Top Track     344
Recently Played     50
Name: count, dtype: int64

In [77]:
# build the user profile
interaction_weights = {
    'Long Top Track' : 1.0,
    'Saved' : 0.8,
    'Recently Played' : 0.75
}

artists_deduped['interaction_weights'] = artists_deduped['interaction_type'].map(interaction_weights).fillna(1.0)
user_profile_taste = (
    artists_deduped.groupby(['artists_id', 'primary_artist_name'])['interaction_weights'].sum().round(2)
    .reset_index(name='total_taste_score')
)

user_profile_taste.sample(10)

,artists_id,primary_artist_name,total_taste_score
77,206,Jacquees,0.8
0,3,2 Chainz,0.8
221,564,Zedd,1.0
68,191,Imagine Dragons,0.8
21,60,Bone Thugs-N-Harmony,0.8
72,199,JACKBOYS,1.8
199,529,Travis Scott,7.2
120,327,NAV,1.6
1,4,21 Savage,3.2
49,153,Fiji Blue,1.8


In [78]:
all_artists.columns

Index(['primary_artist_name', 'artist_listeners', 'artist_playcount', 'tags',
       'similar_artists', 'source'],
      dtype='str')

In [90]:
def generate_reco(user_taste, feature_matrix, all_artists_df, n_items: int = 10, randomize: bool = True):
    full_scores = np.zeros(feature_matrix.shape[1])
    user_artist_indices = []
    
    for _, row in user_taste.iterrows():
        match_idx = all_artists_df[all_artists_df['primary_artist_name'] == row['primary_artist_name']].index
        if len(match_idx) > 0:
            idx = match_idx[0]
            if idx < len(full_scores):
                full_scores[idx] = row['total_taste_score']
                user_artist_indices.append(idx)

    scores = feature_matrix.dot(full_scores)
    scores[user_artist_indices] = -np.inf
    
    if randomize:
        top_pool_size = max(50, n_items * 5)
        top_pool_indices = np.argsort(-scores)[:top_pool_size]

        pool_scores = scores[top_pool_indices]
        pool_scores = np.clip(pool_scores, 0, None)
        
        if pool_scores.sum() > 0:
            probs = pool_scores / pool_scores.sum()
            top_idx = np.random.choice(top_pool_indices, size=n_items, replace=False, p=probs)
        else:
            top_idx = np.argsort(-scores)[:n_items]
    else:
        top_idx = np.argsort(-scores)[:n_items]
    
    return top_idx

recommended_indices = generate_reco(user_profile_taste, combined_feats, all_artists, 10, randomize=True)

In [91]:
all_artists.loc[all_artists.index.isin(recommended_indices), :]

,primary_artist_name,artist_listeners,artist_playcount,tags,similar_artists,source
37,B.B. King,1355639,20976230,"blues, blues rock, jazz, guitar, classic rock,...","albert king, freddie king, buddy guy, albert c...",library
418,Skillet,2312908,87600220,"alternative rock, christian rock, rock, altern...","thousand foot krutch, ashes remain, red, disci...",library
430,Soapdish,89590,1584541,"filipino music, opm","cueshé, silent sanctuary, hale, sponge cola, d...",library
735,Obskür,376898,3563608,"house, techno, irish","chris stussy, josh baker, sidney charles, lost...",candidate
759,PAWSA,457523,5277692,"house, tech house, techno, electronic","mau p, cloonee, mochakk, michael bibi, beltran...",candidate
788,it's murph,265987,3779406,"house, electronic, dance","nimino, john summit, disco lines, noizu, angry...",candidate
901,Dipha Barus,26575,262242,"techno, electronic, indonesia, funk_add_to_lid...","diskoria, lomba sihir, biru baru, skyline, mik...",candidate
1040,TAEYEON,673653,60497182,"k-pop, korean, taeyeon, snsd, pop, why so gorg...","소녀시대, girls' generation-tts, tiffany young, je...",candidate
1043,SISTAR,391200,11968178,"k-pop, korean, sistar, pop, dance, girl group,...","aoa, girl's day, t-ara, miss a, 4minute, sista...",candidate
1256,Ben&Ben,210553,10940042,"pop, folk, opm, filipino pop rock, dance, indi...","december avenue, lola amour, nobita, silent sa...",candidate
